In [1]:
# Все библиотеки

import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.stats import ttest_ind, levene, mannwhitneyu,shapiro, \
    f_oneway, kruskal, chi2_contingency, fisher_exact
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import scikit_posthocs as sp
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import pingouin as pg
import sys
import os
from ydata_profiling import ProfileReport
from matplotlib import rcParams
from matplotlib import font_manager as fm
import matplotlib as mpl
import warnings

mpl.rcParams['font.family'] = 'HSE Sans'

warnings.filterwarnings('ignore')

/Users/ssd/Documents/src/dano_base/base/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Основные графики

Предствленные функции:
- plot_bar_chart -- столбчатая диаграмма
- plot_bar_chart_with_success -- столбчатая диаграмма с долей успеха
- plot_line_chart -- линейная диаграмма
- norm_hist -- Гистограмма
- plot_heatmap -- Красивая Хитмапа

In [12]:
def plot_bar_chart(x, y, title=None, color='#4A90E2', bgcolor='#F5F5F5', figsize=(8, 5)):
    """
    Рисует столбчатую диаграмму с подписями значений над столбцами.

    x : list/array-like — подписи для оси X
    y : list/array-like — высоты столбцов
    title : str — заголовок графика
    color : str — цвет столбцов (hex или имя)
    bgcolor : str — цвет фона графика
    figsize : pair - figsize
    """
    fig, ax = plt.subplots(figsize=figsize)

    # Фон
    fig.patch.set_facecolor(bgcolor)
    ax.set_facecolor(bgcolor)

    # Построение столбцов
    bars = ax.bar(x, y, color=color, edgecolor='black')

    # Подписи над столбцами
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),  # отступ в пикселях
                    textcoords="offset points",
                    ha='center', va='bottom',
                    fontsize=10, color='black')

    # Заголовок
    if title:
        ax.set_title(title, fontsize=14, fontweight='bold')

    # Убираем рамку сверху и справа (минимализм)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()


In [14]:
def plot_line_chart(x, y, title=None, color='#4A90E2', bgcolor='#F5F5F5', figsize=(8, 5), marker='o'):
    """
    Рисует линейную диаграмму с подписями значений в точках.

    x : list/array-like — значения для оси X
    y : list/array-like — значения для оси Y
    title : str — заголовок графика
    color : str — цвет линии (hex или имя)
    bgcolor : str — цвет фона графика
    figsize : tuple — размер графика (ширина, высота)
    marker : str — маркер для точек линии
    """
    fig, ax = plt.subplots(figsize=figsize)

    # Фон
    fig.patch.set_facecolor(bgcolor)
    ax.set_facecolor(bgcolor)

    # Построение линии
    ax.plot(x, y, color=color, marker=marker, linewidth=2)

    # Подписи точек
    for xi, yi in zip(x, y):
        ax.annotate(f'{yi:.2f}',
                    xy=(xi, yi),
                    xytext=(0, 5),  # смещение текста
                    textcoords="offset points",
                    ha='center', va='bottom',
                    fontsize=10, color='black')

    # Заголовок
    if title:
        ax.set_title(title, fontsize=14, fontweight='bold')

    # Убираем рамки сверху и справа
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()


In [17]:
def plot_bar_chart_with_success(
    x, y, success,
    title=None,
    color='#4A90E2',
    success_color='#50E3C2',
    bgcolor='#F5F5F5',
    figsize=(8, 5)
):
    """
    Рисует столбчатую диаграмму с успешной частью у основания.

    x : list/array-like — подписи оси X
    y : list/array-like — общие значения
    success : list/array-like — значения успешных единиц (<= y)
    title : str — заголовок
    color : str — цвет основной части столбца
    success_color : str — цвет успешной части
    bgcolor : str — цвет фона
    figsize : tuple — размер фигуры
    """
    fig, ax = plt.subplots(figsize=figsize)

    # Фон
    fig.patch.set_facecolor(bgcolor)
    ax.set_facecolor(bgcolor)

    # Часть успеха у основания
    success_bars = ax.bar(x, success, color=success_color, edgecolor='black')

    # Остаток над успехом
    remaining = [total - succ for total, succ in zip(y, success)]
    ax.bar(x, remaining, bottom=success, color=color, edgecolor='black')

    # Подписи процентов над успешной частью
    for i, (succ, total, bar) in enumerate(zip(success, y, success_bars)):
        if total > 0:
            percent = succ / total * 100
            ax.annotate(f'{percent:.1f}%',
                        xy=(bar.get_x() + bar.get_width() / 2, succ),
                        xytext=(0, 3),
                        textcoords="offset points",
                        ha='center', va='bottom',
                        fontsize=10, color='black')

    # Подписи общего значения сверху
    for xi, total in zip(x, y):
        ax.annotate(f'{total:.0f}',
                    xy=(xi, total),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom',
                    fontsize=9, color='black', fontweight='bold')

    # Заголовок
    if title:
        ax.set_title(title, fontsize=14, fontweight='bold')

    # Минимализм
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()


In [ ]:
def norm_hist(
    series,
    bins=10,
    color='#4A90E2',
    edgecolor='black',
    bgcolor='#F5F5F5',
    title=None,
    figsize=(8, 5),
    podpisi=True
):
    """
    Рисует красивую гистограмму для Series в стиле минимализма.

    series : pandas.Series — данные
    bins : int — количество корзин
    color : str — цвет столбцов
    edgecolor : str — цвет границ столбцов
    bgcolor : str — цвет фона
    title : str — заголовок
    figsize : tuple — размер графика
    podpisi : bool - делать ли подписи над столбцами
    """
    fig, ax = plt.subplots(figsize=figsize)

    # Фон
    fig.patch.set_facecolor(bgcolor)
    ax.set_facecolor(bgcolor)

    # Строим гистограмму
    counts, bin_edges, patches = ax.hist(series, bins=bins, color=color, edgecolor=edgecolor)

    if podpisi == True:
        # Подписи значений над столбцами
        for count, left, right in zip(counts, bin_edges[:-1], bin_edges[1:]):
            if count > 0:
                ax.annotate(f'{int(count)}',
                            xy=((left + right) / 2, count),
                            xytext=(0, 3),
                            textcoords="offset points",
                            ha='center', va='bottom',
                            fontsize=10, color='black')

    # Заголовок
    if title:
        ax.set_title(title, fontsize=14, fontweight='bold')

    # Минимализм
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()


In [2]:
from matplotlib.colors import Normalize, TwoSlopeNorm


def plot_heatmap(data,
                 x_labels=None,
                 y_labels=None,
                 title=None,
                 cmap='Blues',
                 bgcolor='#F5F5F5',
                 figsize=(8, 6),
                 annot=True,
                 fmt='.2f',
                 cbar_label=None,
                 vmin=None,
                 vmax=None,
                 center=None,
                 linewidths=0.5,
                 linecolor='white'):
    """
    Рисует аккуратную heatmap.

    Параметры (кратко):
    - data : 2D array-like (список списков / numpy array)
    - x_labels, y_labels : подписи для колонок/строк (по умолчанию индексы)
    - title : заголовок графика
    - cmap : цветовая карта (по умолчанию 'Blues' — в твоём синем стиле)
    - bgcolor : цвет фона фигуры
    - figsize : размер фигуры
    - annot : подписывать значения в ячейках
    - fmt : формат подписей (строка форматирования, например '.2f' или 'd')
    - cbar_label : подпись для colorbar
    - vmin, vmax : ручной минимум/максимум для cmap. По умолчанию берутся из данных.
    - center : центр для двухцветной нормировки (если нужно центрировать ноль) — тогда используется TwoSlopeNorm
    - linewidths, linecolor : тонкие линии между ячейками

    Функция корректно обрабатывает NaN (оставляет пустые ячейки) и автоматически выбирает цвет текста
    (чёрный для светлых ячеек, белый для тёмных) по относительной яркости.
    """

    data = np.array(data, dtype=float)
    if data.ndim != 2:
        raise ValueError("data must be 2D (matrix-like)")

    nrows, ncols = data.shape

    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(bgcolor)
    ax.set_facecolor(bgcolor)

    # пределы
    if vmin is None:
        # игнорируем NaN при поиске минимума/максимума
        vmin = np.nanmin(data)
    if vmax is None:
        vmax = np.nanmax(data)

    # нормировка (центрируем, если надо)
    if center is not None:
        norm = TwoSlopeNorm(vcenter=center, vmin=vmin, vmax=vmax)
    else:
        norm = Normalize(vmin=vmin, vmax=vmax)

    im = ax.imshow(data, cmap=cmap, norm=norm, aspect='auto', interpolation='nearest')

    # colorbar
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    if cbar_label:
        cbar.set_label(cbar_label, fontsize=10)

    # подписи осей
    if x_labels is None:
        x_labels = [str(i) for i in range(ncols)]
    if y_labels is None:
        y_labels = [str(i) for i in range(nrows)]

    ax.set_xticks(np.arange(ncols))
    ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(np.arange(nrows))
    ax.set_yticklabels(y_labels, fontsize=9)

    # тонкие линии между ячейками
    ax.set_xticks(np.arange(-0.5, ncols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, nrows, 1), minor=True)
    ax.grid(which='minor', color=linecolor, linestyle='-', linewidth=linewidths)
    ax.tick_params(which='minor', bottom=False, left=False)

    # подписи в ячейках
    if annot:
        # защитимся от деления на ноль при одинаковых vmin/vmax
        denom = (vmax - vmin) if (vmax - vmin) != 0 else 1.0
        for i in range(nrows):
            for j in range(ncols):
                val = data[i, j]
                if np.isnan(val):
                    text = ''
                else:
                    try:
                        text = format(val, fmt)
                    except Exception:
                        # если пользователь передал некорректный формат —fallback
                        text = f"{val:.2f}"

                if text != '':
                    # относительная яркость для выбора цвета текста
                    rel = (val - vmin) / denom
                    text_color = 'white' if rel > 0.5 else 'black'
                    ax.text(j, i, text, ha='center', va='center', fontsize=9, color=text_color)

    # заголовок
    if title:
        ax.set_title(title, fontsize=14, fontweight='bold')

    # минимализм — убираем рамки сверху и справа
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()